# HDB Senior Data Engineer Technical Assessment — Part 1

## Developing Data Pipelines

This notebook is the reviewer-facing walkthrough for the Jan 2012–Dec 2017 HDB resale flat ETL pipeline. Reusable implementation is kept under `src/hdb_etl`; this notebook explains the engineering decisions, executes the pipeline, and surfaces the intended outputs.

**Pipeline:** Raw → Profiling → Jan-2012 reference domains → Validation/Quarantine → Lease calculation → Price anomaly flags → Resale Identifier → Collision handling → HMAC-SHA256 de-identification.


## 1. Key assumptions and engineering decisions

1. **Jan 2012 is the strict authoritative set** for `town`, `flat_type`, `flat_model`, and `storey_range`. Allowed values are derived dynamically; later unseen values are quarantined rather than remapped.
2. **Date semantics change at March 2012.** Jan–Feb 2012 is approval-date based; Mar 2012 onward is registration-date based. The supplied `month` is preserved and `_date_basis` is retained as lineage.
3. **Raw files remain unchanged.** Standardisation/filtering occurs only after reading raw inputs.
4. **`remaining_lease` is source schema evolution, not the assessment calculation.** The assessment lease is derived independently from `lease_commence_date`.
5. **Lease commencement is assumed to be 1 January of the stated commencement year** because the source supplies only a year.
6. **Hard validation failures are quarantined; resale-price anomalies are soft flags.** An unusual transaction is not automatically erroneous.
7. **Duplicate records are treated as duplicate candidates** because the source has no transaction identifier. Subsequent exact matches are quarantined and retained for auditability.
8. **Identifier collisions are resolved deterministically** with suffixes; no accepted record is dropped.
9. **HMAC-SHA256 is used instead of plain SHA-256** so predictable identifiers cannot be precomputed without the secret key.
10. **Idempotency:** record-level run metadata is deterministic from raw file checksums rather than wall-clock timestamps.

See `reports/data_profiling_report.md` for the full rationale and lifecycle findings.


In [1]:
from pathlib import Path
import os
import sys
import secrets
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from hdb_etl.pipeline import run_pipeline
from hdb_etl.extract import file_sha256

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\Ruwei\OneDrive\Desktop\hdb_senior_data_engineer_submission


## 2. Raw inputs and source lineage

The four raw files below overlap the assessment window. They are stored under `data/raw/` unchanged. The older 1990–1999 collection member does not overlap Jan 2012–Dec 2017 and is therefore not an input to this pipeline.

The production extraction module can discover overlapping child datasets from data.gov.sg Collection 189 using collection/dataset metadata and download them programmatically. The raw files are already bundled here so review does not depend on network availability.


In [2]:
raw_files = sorted((PROJECT_ROOT / "data/raw").glob("*.csv"))
raw_manifest = pd.DataFrame([
    {
        "file": p.name,
        "size_mb": round(p.stat().st_size / 1024**2, 2),
        "sha256": file_sha256(p),
    }
    for p in raw_files
])
display(raw_manifest)

,file,size_mb,sha256
0,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",28.01,5f6a72bd7b9120281863beb1fc4c89aa5f4923f00b0f6c52494c4347363f2c36
1,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",2.93,4ff7ce4a4f642fb384d2b75a42e75b0477e50005f4aaa9989591a7a92507d185
2,"Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",3.97,6a16e2dc78f70048ec1a522b59a9727581c36e45e02022641c53e755a7657c72
3,Resale flat prices based on registration date from Jan-2017 onwards.csv,22.83,945e09d75efb2eec1ab1618ce369c0d8eb885ef34158e29426f1af141d4000e4


## 3. Execute the ETL pipeline

The notebook calls the reusable pipeline rather than reimplementing business logic in notebook cells. The notebook creates an **ephemeral in-memory HMAC secret** for this demonstration run. It is never printed or persisted. In a production deployment, a stable secret must be supplied externally (for example through a secret manager/environment variable) and must never be committed to Git.


In [3]:
EPHEMERAL_SECRET = secrets.token_urlsafe(32)
result = run_pipeline(
    project_root=PROJECT_ROOT,
    hash_secret=EPHEMERAL_SECRET,
    write_outputs=True,
)

summary = pd.DataFrame(result["summary"].items(), columns=["metric", "rows"])
display(summary)

,metric,rows
0,raw_rows_across_selected_files,701039
1,in_scope_rows,113053
2,cleaned_rows,104783
3,quarantined_rows,8270
4,transformed_rows,104783
5,hashed_rows,104783
6,price_anomaly_rows,2110


## 4. Reconciliation checks

A core engineering requirement is that no in-scope row disappears silently. Every row must end either in the accepted path or the Quarantined bucket; subsequent stages must preserve accepted row counts.


In [4]:
assert result["summary"]["cleaned_rows"] + result["summary"]["quarantined_rows"] == result["summary"]["in_scope_rows"]
assert result["summary"]["transformed_rows"] == result["summary"]["cleaned_rows"]
assert result["summary"]["hashed_rows"] == result["summary"]["cleaned_rows"]
assert result["transformed"]["resale_identifier"].is_unique
assert result["hashed"]["hashed_resale_identifier"].is_unique
print("PASS: row reconciliation and identifier uniqueness checks succeeded.")

PASS: row reconciliation and identifier uniqueness checks succeeded.


## 5. Jan 2012 authoritative domains

The reference values are extracted from Jan 2012 at runtime instead of hardcoding categories. Jan 2012 to act as the **strict** authoritative set.


In [5]:
domain_summary = pd.DataFrame([
    {"column": column, "distinct_authoritative_values": len(values), "sample": sorted(map(str, values))[:12]}
    for column, values in result["reference_domains"].items()
])
display(domain_summary)

,column,distinct_authoritative_values,sample
0,town,26,"[ANG MO KIO, BEDOK, BISHAN, BUKIT BATOK, BUKIT MERAH, BUKIT PANJANG, BUKIT T..."
1,flat_type,7,"[1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION]"
2,flat_model,13,"[Adjoined flat, Apartment, Improved, Maisonette, Model A, Model A-Maisonette..."
3,storey_range,12,"[01 TO 03, 04 TO 06, 07 TO 09, 10 TO 12, 13 TO 15, 16 TO 18, 19 TO 21, 22 TO..."


## 6. Raw/in-scope profiling

The union schema contains a later `remaining_lease` field that is not present in the earlier files. Its historical nulls are therefore interpreted as schema evolution rather than automatically classified as bad records.


In [6]:
display(result["raw_profile"])
print("In-scope source-date basis:")
display(result["master"]["_date_basis"].value_counts().rename_axis("date_basis").reset_index(name="rows"))

,column,dtype,rows,null_count,null_pct,distinct_count,min,max,mean,median
0,month,object,113053,0,0.000,72,NaN,NaN,NaN,NaN
1,town,object,113053,0,0.000,26,NaN,NaN,NaN,NaN
2,flat_type,object,113053,0,0.000,7,NaN,NaN,NaN,NaN
3,block,object,113053,0,0.000,2193,NaN,NaN,NaN,NaN
4,street_name,object,113053,0,0.000,526,NaN,NaN,NaN,NaN
5,storey_range,object,113053,0,0.000,25,NaN,NaN,NaN,NaN
6,floor_area_sqm,float64,113053,0,0.000,169,31.0,280.0,96.784,96.0
7,flat_model,object,113053,0,0.000,20,NaN,NaN,NaN,NaN
8,lease_commence_date,int64,113053,0,0.000,50,1966.0,2015.0,1990.554,1988.0
9,resale_price,float64,113053,0,0.000,2865,175000.0,1180000.0,449659.946,425000.0


In-scope source-date basis:


,date_basis,rows
0,registration_date,109865
1,approval_date,3188


## 7. Validation and Quarantined bucket

Hard validation checks include the four Jan-2012 authoritative domains, required values, valid month/window, positive numeric values, plausible lease commencement year, and duplicate candidates. Validation reasons are retained rather than silently dropping records.

A notable interpretation point is that later `flat_model` and `storey_range` values can be legitimate domain evolution while still failing the assessment's strict Jan-2012 rule. The profiling report explicitly separates that from proven source corruption.


In [7]:
display(result["quarantine_reason_counts"])

q = result["quarantined"]
q_by_year = q.assign(year=q["month"].str[:4]).groupby("year").size().rename("rows").reset_index()
display(q_by_year)

display(q[["month", "town", "flat_type", "flat_model", "storey_range", "validation_errors", "_source_file", "_source_row_number"]].head(15))

,validation_rule,row_count
0,INVALID_STOREY_RANGE,7050
1,INVALID_FLAT_MODEL,870
2,DUPLICATE_RECORD,451


,year,rows
0,2012,6860
1,2013,27
2,2014,315
3,2015,192
4,2016,283
5,2017,593


,month,town,flat_type,flat_model,storey_range,validation_errors,_source_file,_source_row_number
2946,2012-02,TOA PAYOH,5 ROOM,Improved,37 TO 39,INVALID_STOREY_RANGE,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",369411
3103,2012-02,YISHUN,3 ROOM,New Generation,01 TO 03,DUPLICATE_RECORD,"Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",369568
3528,2015-01,CENTRAL AREA,4 ROOM,Type S1,19 TO 21,INVALID_FLAT_MODEL,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",342
3529,2015-01,CENTRAL AREA,4 ROOM,Type S1,22 TO 24,INVALID_FLAT_MODEL,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",343
3530,2015-01,CENTRAL AREA,4 ROOM,Type S1,34 TO 36,INVALID_FLAT_MODEL,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",344
3531,2015-01,CENTRAL AREA,4 ROOM,Type S1,22 TO 24,INVALID_FLAT_MODEL,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",345
3532,2015-01,CENTRAL AREA,4 ROOM,Type S1,22 TO 24,INVALID_FLAT_MODEL,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",346
3533,2015-01,CENTRAL AREA,4 ROOM,Type S1,34 TO 36,INVALID_FLAT_MODEL,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",347
3534,2015-01,CENTRAL AREA,4 ROOM,Type S1,46 TO 48,INVALID_FLAT_MODEL|INVALID_STOREY_RANGE,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",348
3535,2015-01,CENTRAL AREA,4 ROOM,Type S1,46 TO 48,INVALID_FLAT_MODEL|INVALID_STOREY_RANGE,"Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",349


## 8. Remaining lease as at 1 Jan 2027

The source's `lease_commence_date` contains only a year. The pipeline assumes 1 January of that year, adds 99 years, and computes remaining whole years/months at 1 Jan 2027. Assumption as exact month/day precision is not available from this source field.


In [8]:
lease_cols = ["lease_commence_date", "remaining_lease_years_2027", "remaining_lease_months_2027", "remaining_lease_as_of_2027"]
display(result["cleaned"][lease_cols].drop_duplicates().sort_values("lease_commence_date").head(15))

,lease_commence_date,remaining_lease_years_2027,remaining_lease_months_2027,remaining_lease_as_of_2027
6111,1966,38,0,38 years 0 months
284,1967,39,0,39 years 0 months
1265,1968,40,0,40 years 0 months
281,1969,41,0,41 years 0 months
690,1970,42,0,42 years 0 months
280,1971,43,0,43 years 0 months
513,1972,44,0,44 years 0 months
179,1973,45,0,45 years 0 months
87,1974,46,0,46 years 0 months
263,1975,47,0,47 years 0 months


## 9. Potential resale-price anomalies

A transparent grouped-IQR rule is applied within `year + town + flat_type` cohorts. Groups with fewer than 20 records are not flagged due to insufficient comparison data. Outliers stay in the Cleaned dataset with an explanatory flag rather than being automatically quarantined.


In [9]:
c = result["cleaned"]
anomaly_summary = c["resale_price_anomaly_reason"].value_counts().rename_axis("reason").reset_index(name="rows")
display(anomaly_summary)

display(c.loc[c["resale_price_anomaly_flag"], [
    "month", "town", "flat_type", "block", "resale_price",
    "price_lower_bound", "price_upper_bound", "resale_price_anomaly_reason"
]].head(15))

,reason,rows
0,WITHIN_IQR_BOUND,101521
1,ABOVE_IQR_BOUND,1967
2,INSUFFICIENT_GROUP_SIZE,1152
3,BELOW_IQR_BOUND,143


,month,town,flat_type,block,resale_price,price_lower_bound,price_upper_bound,resale_price_anomaly_reason
4,2012-01,ANG MO KIO,2 ROOM,174,226000.0,240500.0,292500.0,BELOW_IQR_BOUND
266,2012-01,BUKIT MERAH,2 ROOM,44,442000.0,233000.0,345000.0,ABOVE_IQR_BOUND
293,2012-01,BUKIT MERAH,3 ROOM,24,575000.0,255500.0,499500.0,ABOVE_IQR_BOUND
357,2012-01,BUKIT PANJANG,4 ROOM,431,330000.0,332000.0,500000.0,BELOW_IQR_BOUND
453,2012-01,CHOA CHU KANG,5 ROOM,789,585000.0,405000.0,573000.0,ABOVE_IQR_BOUND
460,2012-01,CHOA CHU KANG,5 ROOM,632,600000.0,405000.0,573000.0,ABOVE_IQR_BOUND
461,2012-01,CHOA CHU KANG,5 ROOM,633,620000.0,405000.0,573000.0,ABOVE_IQR_BOUND
517,2012-01,GEYLANG,3 ROOM,3,435000.0,225000.0,425000.0,ABOVE_IQR_BOUND
555,2012-01,HOUGANG,3 ROOM,7,260000.0,285000.0,405000.0,BELOW_IQR_BOUND
565,2012-01,HOUGANG,3 ROOM,210,420000.0,285000.0,405000.0,ABOVE_IQR_BOUND


## 10. Resale Identifier and deterministic collision handling

The base identifier follows the assessment rule. Because the key compresses multiple attributes, collisions are expected. Collision groups receive deterministic `-001`, `-002`, ... suffixes ordered by source lineage. Non-colliding identifiers remain unchanged.


In [10]:
t = result["transformed"]
collision_metrics = {
    "rows_in_collision_groups": int((t["identifier_collision_size"] > 1).sum()),
    "distinct_colliding_base_identifiers": int(t.loc[t["identifier_collision_size"] > 1, "base_resale_identifier"].nunique()),
    "largest_collision_group": int(t["identifier_collision_size"].max()),
    "final_identifier_unique": bool(t["resale_identifier"].is_unique),
}
display(pd.DataFrame(collision_metrics.items(), columns=["metric", "value"]))
display(t.loc[t["identifier_collision_size"] > 1, [
    "month", "town", "flat_type", "block", "group_avg_resale_price",
    "base_resale_identifier", "identifier_collision_sequence", "resale_identifier"
]].head(15))

,metric,value
0,rows_in_collision_groups,23639
1,distinct_colliding_base_identifiers,10687
2,largest_collision_group,14
3,final_identifier_unique,True


,month,town,flat_type,block,group_avg_resale_price,base_resale_identifier,identifier_collision_sequence,resale_identifier
1,2012-01,ANG MO KIO,2 ROOM,314,256966.666667,S3142561201A,1,S3142561201A-001
2,2012-01,ANG MO KIO,2 ROOM,314,256966.666667,S3142561201A,2,S3142561201A-002
15,2012-01,ANG MO KIO,3 ROOM,558,347535.449524,S5583471201A,1,S5583471201A-001
21,2012-01,ANG MO KIO,3 ROOM,550,347535.449524,S5503471201A,1,S5503471201A-001
22,2012-01,ANG MO KIO,3 ROOM,550,347535.449524,S5503471201A,2,S5503471201A-002
24,2012-01,ANG MO KIO,3 ROOM,558,347535.449524,S5583471201A,2,S5583471201A-002
30,2012-01,ANG MO KIO,3 ROOM,256,347535.449524,S2563471201A,1,S2563471201A-001
31,2012-01,ANG MO KIO,3 ROOM,256,347535.449524,S2563471201A,2,S2563471201A-002
42,2012-01,ANG MO KIO,3 ROOM,633,347535.449524,S6333471201A,1,S6333471201A-001
43,2012-01,ANG MO KIO,3 ROOM,633,347535.449524,S6333471201A,2,S6333471201A-002


## 11. Irreversible de-identification

The Hashed output uses keyed HMAC-SHA256. The clear-text `resale_identifier` and `base_resale_identifier` are removed from that output. Using a secret key is preferable to plain SHA-256 for a structured/predictable identifier because an attacker cannot reproduce candidate hashes without the secret.


In [11]:
h = result["hashed"]
print("Hashed identifier unique:", h["hashed_resale_identifier"].is_unique)
print("Clear-text resale_identifier present:", "resale_identifier" in h.columns)
display(h[["month", "town", "flat_type", "block", "hashed_resale_identifier"]].head(10))

Hashed identifier unique: True
Clear-text resale_identifier present: False


,month,town,flat_type,block,hashed_resale_identifier
0,2012-01,ANG MO KIO,2 ROOM,406,d2762693ad2970d8995de69d63c88b0ed75bdd43458abce717e6159dc56b66dd
1,2012-01,ANG MO KIO,2 ROOM,314,e19917fdafab0c086a71f14349dbfa1ecbe91f76e6dbd5a9755f725d65cfc1ed
2,2012-01,ANG MO KIO,2 ROOM,314,162e4f2c190a647475aba74c8964fa12bfb7868ea4e333275fa4f1533976e28c
3,2012-01,ANG MO KIO,2 ROOM,170,82500bdaa5ba0d1e6855a76a278bf6f49e45b77a2daff7729319150964bea9c7
4,2012-01,ANG MO KIO,2 ROOM,174,9334af2a6a3eed22d29179bda060b756f332d4bebd968494e47153956713881a
5,2012-01,ANG MO KIO,2 ROOM,508,ab2979c1945386a1ef9b74ba633590549fb3119267e7367e521b8531b5bda6bb
6,2012-01,ANG MO KIO,3 ROOM,174,a91bd38807a8c4aba603e4a9aea3b3bd759f8ef51edbd708323d8db41a86d5c4
7,2012-01,ANG MO KIO,3 ROOM,216,840a4ae60e9b94d16041b7e9d33f1eba247e2c68221aca5840362ecc985ba8c9
8,2012-01,ANG MO KIO,3 ROOM,332,481f491bf8dfb12cf3501e005c81511f29158a0b33f3851e2371368b5dffb77e
9,2012-01,ANG MO KIO,3 ROOM,418,ecde1799a208064e5fd635d4e6e99215ea56f87555cbf3c5f271b319ff9dcab9


## 12. Mandatory outputs produced

The execution above writes the five required groups:

- **Raw:** `data/raw/` — source files unchanged
- **Cleaned:** `data/cleaned/hdb_resale_cleaned.csv`
- **Transformed:** `data/transformed/hdb_resale_transformed.csv`
- **Quarantined:** `data/quarantined/hdb_resale_quarantined.csv`
- **Hashed:** `data/hashed/hdb_resale_hashed.csv`

The detailed lifecycle findings and quarantine interpretation are in `reports/data_profiling_report.md`. Setup/execution instructions are in the root `README.md`.


In [12]:
output_files = [
    PROJECT_ROOT / "data/cleaned/hdb_resale_cleaned.csv",
    PROJECT_ROOT / "data/transformed/hdb_resale_transformed.csv",
    PROJECT_ROOT / "data/quarantined/hdb_resale_quarantined.csv",
    PROJECT_ROOT / "data/hashed/hdb_resale_hashed.csv",
    PROJECT_ROOT / "reports/data_profiling_report.md",
]
display(pd.DataFrame([
    {"output": str(p.relative_to(PROJECT_ROOT)), "exists": p.exists(), "size_mb": round(p.stat().st_size / 1024**2, 2) if p.exists() else None}
    for p in output_files
]))

,output,exists,size_mb
0,data\cleaned\hdb_resale_cleaned.csv,True,31.41
1,data\transformed\hdb_resale_transformed.csv,True,36.06
2,data\quarantined\hdb_resale_quarantined.csv,True,1.89
3,data\hashed\hdb_resale_hashed.csv,True,39.87
4,reports\data_profiling_report.md,True,0.01
